In [ ]:
import os
import argparse
import joblib
import numpy as np
import pandas as pd


WARMUP = 1000
WINDOWS = [5, 10, 20, 50, 100]
LAGS = [1, 2, 3, 5, 10, 20]


def load_csv(path):
    df = pd.read_csv(path)
    df["sending_time"] = df["sending_time"].astype("int64")
    df["arrival_time"] = df["arrival_time"].astype("int64")
    return df


def add_delay(df):
    df = df.copy()
    df["delay"] = df["arrival_time"] - df["sending_time"]
    return df


def add_own_features(df):
    df = df.copy()
    df["row_id"] = np.arange(len(df))

    df = df.sort_values("arrival_time").reset_index(drop=True)

    s = df["delay"]
    shifted = s.shift(1)

    df["message_type_enc"] = (df["message_type"] == "MarketChange").astype(int)

    for k in LAGS:
        df[f"own_lag_{k}"] = s.shift(k)

    for w in WINDOWS:
        df[f"own_mean_{w}"] = shifted.rolling(w, min_periods=1).mean()
        df[f"own_std_{w}"] = shifted.rolling(w, min_periods=2).std()
        df[f"own_median_{w}"] = shifted.rolling(w, min_periods=1).median()
        df[f"own_max_{w}"] = shifted.rolling(w, min_periods=1).max()
        df[f"own_min_{w}"] = shifted.rolling(w, min_periods=1).min()

    df["own_arrival_gap"] = df["arrival_time"].diff()
    df["own_gap_mean_10"] = df["own_arrival_gap"].rolling(10, min_periods=1).mean()

    for w in WINDOWS:
        df[f"own_spike_ratio_{w}"] = df["own_lag_1"] / (df[f"own_mean_{w}"] + 1e-6)

    df["row_number_in_day"] = np.arange(1, len(df) + 1)

    return df.fillna(0)


def make_state_features_for_other_channel(df):
    df = df.sort_values("arrival_time").reset_index(drop=True).copy()

    s = df["delay"]

    state = pd.DataFrame()
    state["arrival_time"] = df["arrival_time"]

    for k in LAGS:
        state[f"other_lag_{k}"] = s.shift(k - 1)

    for w in WINDOWS:
        state[f"other_mean_{w}"] = s.rolling(w, min_periods=1).mean()
        state[f"other_std_{w}"] = s.rolling(w, min_periods=2).std()
        state[f"other_median_{w}"] = s.rolling(w, min_periods=1).median()
        state[f"other_max_{w}"] = s.rolling(w, min_periods=1).max()
        state[f"other_min_{w}"] = s.rolling(w, min_periods=1).min()

    for w in WINDOWS:
        state[f"other_spike_ratio_{w}"] = state["other_lag_1"] / (
            state[f"other_mean_{w}"] + 1e-6
        )

    return state.fillna(0)


def build_features_for_one_day(df_a, df_b):
    df_a = add_delay(df_a)
    df_b = add_delay(df_b)

    df_a = add_own_features(df_a)
    df_b = add_own_features(df_b)

    df_a["stock"] = "A"
    df_b["stock"] = "B"

    first_arrival_time = min(
        df_a["arrival_time"].min(),
        df_b["arrival_time"].min()
    )

    for df in [df_a, df_b]:
        df["seconds_from_day_start"] = (
            df["arrival_time"] - first_arrival_time
        ) / 1_000_000_000

        df["minutes_from_day_start"] = df["seconds_from_day_start"] / 60

    state_a = make_state_features_for_other_channel(df_a)
    state_b = make_state_features_for_other_channel(df_b)

    df_a = pd.merge_asof(
        df_a.sort_values("arrival_time"),
        state_b.sort_values("arrival_time"),
        on="arrival_time",
        direction="backward",
        allow_exact_matches=False
    )

    df_b = pd.merge_asof(
        df_b.sort_values("arrival_time"),
        state_a.sort_values("arrival_time"),
        on="arrival_time",
        direction="backward",
        allow_exact_matches=False
    )

    df_a = df_a.fillna(0)
    df_b = df_b.fillna(0)

    return df_a, df_b


def predict_stock(df_features, model, feature_list):
    X = df_features[feature_list].astype("float32")

    pred_log = model.predict(X)
    pred = np.expm1(pred_log)
    pred = np.maximum(pred, 0)

    return pred.round().astype("int64")


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--stock_a", required=True)
    parser.add_argument("--stock_b", required=True)
    parser.add_argument("--output_dir", required=True)

    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    script_dir = os.path.dirname(os.path.abspath(__file__))

    model_A_path = os.path.join(script_dir, "model_A_xgboost.pkl")
    model_B_path = os.path.join(script_dir, "model_B_lightgbm.pkl")

    features_A_path = os.path.join(script_dir, "features_A.pkl")
    features_B_path = os.path.join(script_dir, "features_B.pkl")

    model_A = joblib.load(model_A_path)
    model_B = joblib.load(model_B_path)

    features_A = joblib.load(features_A_path)
    features_B = joblib.load(features_B_path)

    # Safer if evaluator has no GPU
    try:
        model_A.set_params(device="cpu")
    except Exception:
        pass

    df_a_raw = load_csv(args.stock_a)
    df_b_raw = load_csv(args.stock_b)

    features_a, features_b = build_features_for_one_day(df_a_raw, df_b_raw)

    pred_a = predict_stock(features_a, model_A, features_A)
    pred_b = predict_stock(features_b, model_B, features_B)

    output_a = features_a[["row_id", "sending_time", "arrival_time"]].copy()
    output_a["prediction"] = pred_a
    output_a = output_a.sort_values("row_id")
    output_a = output_a[["sending_time", "arrival_time", "prediction"]]

    output_b = features_b[["row_id", "sending_time", "arrival_time"]].copy()
    output_b["prediction"] = pred_b
    output_b = output_b.sort_values("row_id")
    output_b = output_b[["sending_time", "arrival_time", "prediction"]]

    output_a.to_csv(
        os.path.join(args.output_dir, "predictions_Stock_A.csv"),
        index=False
    )

    output_b.to_csv(
        os.path.join(args.output_dir, "predictions_Stock_B.csv"),
        index=False
    )

    print("Saved predictions_Stock_A.csv")
    print("Saved predictions_Stock_B.csv")


if __name__ == "__main__":
    main()